# One-vs-one LightGBM

We build 2 LightGBM cpu models, the first is fit vs at-risk and the second is unhealthy vs at-risk. In other words, compare two minor classes vs major class.
Then we recalculate oof and test probabilities for all 3 classes using conditional probability definition.

We don't use sample weights, because it leads to overfitting for highly imbalanced classes. Instead, we tune probability calibration for balanced accuracy in the end of notebook.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, TargetEncoder, KBinsDiscretizer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score, recall_score, roc_curve
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
import gc
import os
from itertools import combinations
from tqdm import tqdm
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMRegressor,LGBMClassifier,log_evaluation,early_stopping
try:
    from cuml.ensemble import RandomForestClassifier
    from cuml.neighbors import KNeighborsClassifier
    from cuml.preprocessing import StandardScaler
    from cuml.pipeline import Pipeline
    import cuml
    cuml.set_global_output_type('numpy')
except:
    pass

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

ID = 'id'
TARGET = 'health_condition'
NUMS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake']
CATS = ['diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender']

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)
SEED_LIST = [60, 0, 2809]
ADD_EXT = False
# Only cpu lightgbm
MODEL_TYPES_1 = ['lgb']
MODEL_TYPES_2 = ['lgb']

def metric(y_true, y_pred):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)

In [ ]:
PATH = '/kaggle/input/competitions/playground-series-s6e7'
train = pd.read_csv(f'{PATH}/train.csv', index_col='id')
test = pd.read_csv(f'{PATH}/test.csv', index_col='id')
train[TARGET] = train[TARGET].map({'at-risk': 0, 'fit': 1, 'unhealthy': 2})

## 2. Feature Engineering

In [ ]:
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
num_cols = train.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = [col for col in cat_cols if col not in [ID, TARGET]]
num_cols = [col for col in num_cols if col not in [ID, TARGET]]
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
def feature_engineering(df, fit=False):
    # Categorize string cats
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Discretize numericals
    # bin_config = {'delta': [100, 500]}
    # for col, bins_list in bin_config.items():
    #     for n_bins in bins_list:
    #         for strategy in ['quantile']:
    #             bin_name = f"{col}_{n_bins}_{strategy}_bin_"
    #             if fit:
    #                 kb = KBinsDiscretizer(
    #                     n_bins=n_bins,
    #                     encode='ordinal',
    #                     strategy=strategy,
    #                     subsample=None
    #                 )
    #                 binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
    #                 category_map[bin_name] = kb
    #             else:
    #                 kb = category_map[bin_name]
    #                 binned = kb.transform(df[[col]]).ravel().astype('int32')
    #             df[bin_name] = binned
    #             df[bin_name] = df[bin_name].astype('category')
    return df

comb = feature_engineering(pd.concat([train.drop(columns=TARGET), test], ignore_index=True), fit=True)
y = train[TARGET].copy()
test = comb.iloc[len(y):].reset_index(drop=True)
train = comb.iloc[:len(y)].reset_index(drop=True)
train[TARGET] = y.copy()
del comb 
gc.collect()
new_cat_cols = [col for col in train.columns if col.endswith('_')]
new_num_cols = [col for col in train.columns if col.startswith('_')]

In [ ]:
DROP=[c for c in test.columns if test[c].nunique()==1]
print(f"DROP:{DROP}")
train.drop(DROP,axis=1,inplace=True)
test.drop(DROP,axis=1,inplace=True)
new_cat_cols = [col for col in new_cat_cols if col not in DROP]

CATEGORY=CATS+[c for c in test.columns if 'digit' in c]
for c in CATEGORY:

    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))
    

FEATURES=CATEGORY+NUMS

## 3. Pair target encoding

In [ ]:
TE_columns = []

columns = NUMS + CATS
for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        combined = pd.concat([train[name], test[name]], ignore_index=True)
        combined, _ = combined.factorize()
        if pd.Series(combined).nunique() > len(combined) // 2:
            train = train.drop(name, axis=1)
            test = test.drop(name, axis=1)
            continue
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

In [ ]:
cat_cols = CATS + new_cat_cols

target_mapping = {'at-risk': 0, 'fit': 1, 'unhealthy': 2}
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
classes = sorted(target_mapping.keys())
train_full = train.copy()
test_df = test.copy().reset_index(drop=True)
del train, test
gc.collect()
train_full['id'] = train_full.index
train_full[TARGET] = train_full[TARGET].map(reverse_mapping)
submission = pd.read_csv(f'{PATH}/sample_submission.csv')
test_df['id'] = submission['id']

feature_cols = [c for c in train_full.columns if c not in ['id', TARGET] and train_full[c].dtype != 'object']
feature_cols = cat_cols + [c for c in feature_cols if c not in cat_cols]

print(f'Total features: {len(feature_cols)}')
print(f'Engineered features: {[c for c in feature_cols if c not in CATS + NUMS]}')

In [ ]:
y = train_full[TARGET].map(target_mapping).values

for col in cat_cols:
    combined_cats = pd.concat([train_full[col], test_df[col]]).astype(str).unique()
    train_full[col] = pd.Categorical(train_full[col].astype(str), categories=combined_cats)
    test_df[col] = pd.Categorical(test_df[col].astype(str), categories=combined_cats)

features = train_full[feature_cols].copy()
test_features = test_df[feature_cols].copy()
test_ids = test_df['id'].values

for c in cat_cols:
    features[c] = features[c].cat.codes.astype('int32')
    test_features[c] = test_features[c].cat.codes.astype('int32')

In [ ]:
y_0 = (y == 0).astype('int')
y_1 = (y == 1).astype('int')
y_2 = (y == 2).astype('int')
target = pd.DataFrame({'at-risk': y_0, 'fit': y_1, 'unhealthy': y_2})

## 4. Model params

In [ ]:
def to_logits(df):
    df_col = df.columns
    epsilon = 1e-7
    preds_x = np.clip(df, epsilon, 1 - epsilon)
    logits = np.log(preds_x / (1 - preds_x))
    preds_x = pd.DataFrame(logits)
    preds_x.columns = df_col
    return preds_x

def balanced_accuracy():
    def xgb_balanced_accuracy(y_true, y_pred):
        n_classes = 2
        y_pred_labels = np.argmax(y_pred.reshape(-1, n_classes), axis=1)
        return balanced_accuracy_score(y_true.astype(int), y_pred_labels)

    xgb_balanced_accuracy.__name__ = 'bal_ACC'
    return xgb_balanced_accuracy

params_xgb = {
    'subsample': 0.8747340624732572,
    'colsample_bytree': 0.5,
    'max_depth': 4,
    'learning_rate': 0.02,
    'max_bin': 1024,
    'tree_method': 'hist',
    'device': 'cuda',
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42
}

params_cat = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 5000,
    'learning_rate': 0.03,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_seed': 42,
    'early_stopping_rounds': 1000,
    'task_type': 'GPU',
    'devices': '0'
}

params_lgb = {
    'random_state': 60,
    'feature_pre_filter': False,
    'verbose': -1,
    'n_estimators': 20000,
    'learning_rate': 0.05,
    'max_depth': 3,
    'min_child_samples': 63,
    'subsample': 0.812763123433567,
    'colsample_bytree': 0.8029300829885024, 
    'num_leaves': 247, 
    'reg_alpha': 0.07094285437903122,
    'reg_lambda': 0.033039097703242495,
    'max_bin': 32000,
}

params_hgb = {
    'max_iter': 20000,
    'random_state': 60,
    'early_stopping': True,
    'categorical_features': "from_dtype",
    'learning_rate': 0.05,
    'loss': 'log_loss',
    'l2_regularization': 1.293878083737259e-05,
    'max_depth': 3, 
    'max_leaf_nodes': 225,
    'min_samples_leaf': 20,
    'tol': 1e-7
 }

In [ ]:
!pip install -q pytabkit

In [ ]:
import random
import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)

In [ ]:
params_tabm = {
    'device': 'cuda',
    'val_metric_name': '1-auc_ovr',
    'random_state': 42,
    'verbosity': 0,
    'tabm_k': 32,
    'num_emb_type': 'pwl',
    'd_embedding': 12,
    'batch_size': 64,
    'lr': 3e-3,
    'weight_decay': 2e-2,
    'n_epochs': 3,
    'dropout': 0.1,
    'd_block': 512,
    'n_blocks': 3
}

In [ ]:
params_realmlp = {
    'val_metric_name':      '1-auc_ovr',
    'n_ens':                8,
    'embedding_size':       8,
    'max_one_hot_cat_size': 8,
    'hidden_sizes':         [512, 512, 512],
    'p_drop':               0.05,
    'p_drop_sched':         'expm4t',
    'act':                  'silu',
    'add_front_scale':      True,

    'plr_hidden_1':  20,
    'plr_hidden_2':  4,
    'plr_sigma':     5.0,
    'plr_act_name': 'prelu',
    'plr_lr_factor': 0.093,

    'lr':                    0.01,
    'mom':                   0.9,
    'sq_mom':                0.98,
    'lr_sched':              'flat_cos',
    'first_layer_lr_factor': 1.0,
    'scale_lr_factor':       10.0,
    'bias_lr_factor':        0.1,
    'wd':                    0.013,
    'bias_wd_factor':        0.5,

    'ls_eps':       0.04,
    'ls_eps_sched': 'cos',

    'tfms': ['median_center', 'robust_scale'],

    'n_epochs':   6,
    'batch_size': 256,
    'verbosity':  2,

    'use_early_stopping':                     False,
    'early_stopping_additive_patience':       10,
    'early_stopping_multiplicative_patience': 1,

    'device':       'cuda',
    'random_state': 42,
}

In [ ]:
def run(model_type, target_name, drop_name, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    target_id = target_mapping[target_name]
    drop_id = None
    if drop_name:
        drop_id = target_mapping[drop_name]
    i = target_id
    target_name = reverse_mapping[target_id]
    for fold, (tr_idx, val_idx) in enumerate(skf.split(features, target[target_name])):
        X_train = features.iloc[tr_idx].reset_index(drop=True)
        X_val = features.iloc[val_idx].reset_index(drop=True)
        X_val_full = features.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        # masking third class
        if drop_id:
            drop_name = reverse_mapping[drop_id]
            train_mask_2 = (train_target[drop_name] == 0)
            val_mask_2 = (val_target[drop_name] == 0)
        else:
            train_mask_2 = (train_target[target_name] >= 0)
            val_mask_2 = (val_target[target_name] >= 0)
        X_train = X_train[train_mask_2]
        train_target = train_target[train_mask_2]
        X_val = X_val[val_mask_2]
        val_target = val_target[val_mask_2]
 
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_features.copy()
    
        if TE_columns:
            encoder = TargetEncoder(cv=5, random_state=seed, smooth='auto')
            result = pd.DataFrame(encoder.fit_transform(X_train[TE_columns], y[tr_idx][train_mask_2]))
            X_train = pd.concat([result.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
            result = pd.DataFrame(encoder.transform(X_val[TE_columns]))
            X_val = pd.concat([result.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
            result = pd.DataFrame(encoder.transform(X_val_full[TE_columns]))
            X_val_full = pd.concat([result.reset_index(drop=True), X_val_full.reset_index(drop=True)], axis=1)
            result = pd.DataFrame(encoder.transform(X_test[TE_columns]))
            X_test = pd.concat([result.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
            X_train = X_train.drop(TE_columns, axis=1)
            X_val = X_val.drop(TE_columns, axis=1)
            X_val_full = X_val_full.drop(TE_columns, axis=1)
            X_test = X_test.drop(TE_columns, axis=1)
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])
        
        if model_type == 'tabm':
            model = TabM_D_Classifier(**params_tabm)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i], cat_col_names=cat_cols)
        elif model_type == 'realmlp':
            model = RealMLP_TD_Classifier(**params_realmlp)
            model.fit(X_train, train_target.iloc[:, i], X_val, val_target.iloc[:, i], cat_col_names=cat_cols)
        elif model_type == 'xgb':
            model = XGBClassifier(
                **params_xgb,
                n_estimators=20000,
                early_stopping_rounds=500,
            )

            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=[(X_val, val_target.iloc[:, i])],
                verbose=1000
            )
        elif model_type == 'cat':
            model = CatBoostClassifier(**params_cat)
    
            model.fit(
                X_train, train_target.iloc[:, i],
                eval_set=(X_val, val_target.iloc[:, i]),
                verbose=500
            )
        elif model_type == 'lr':
            model = LogisticRegressionCV(max_iter=10000, random_state=42, Cs=10, scoring='roc_auc', cv=5)

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'lgb':
            model=LGBMClassifier(**params_lgb)
    
            model.fit(X_train, y_train,eval_set=[(X_val, y_val)],
                      eval_metric='auc',
                      callbacks=[log_evaluation(500),early_stopping(250)])
        elif model_type == 'rf':
            model = RandomForestClassifier(n_estimators=500, max_depth=12, random_state=42, max_features=0.3, output_type='numpy')

            model.fit(
                X_train, train_target.iloc[:, i]
            )
        elif model_type == 'knn':
            X_train = X_train.astype('float32')
            X_val = X_val.astype('float32')
            X_test = X_test.astype('float32')
            model = Pipeline([
                ('scaler', StandardScaler()), 
                ('knn', KNeighborsClassifier(n_neighbors=200, weights='distance'))
            ])
            model.fit(X_train, train_target.iloc[:, i])
        elif model_type == 'hgb':
            model = HistGradientBoostingClassifier(**params_hgb)
            model.fit(X_train, train_target.iloc[:, i], X_val=X_val, y_val=val_target.iloc[:, i])

        p_val = model.predict_proba(X_val)[:, 1]
        p_val_full = model.predict_proba(X_val_full)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]

        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_1_t = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_1_t.to_parquet(f'oof_preds_{target_id}_{model_type}_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    test_preds_1_t = pd.DataFrame(p_test_x_final, columns=target.columns)
    test_preds_1_t['id'] = test_ids
    test_preds_1_t.to_parquet(f'test_preds_{target_id}_{model_type}_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return h

In [ ]:
def run_blend(preds_tmp, test_tmp, target_name, drop_name, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    p_val_x = np.zeros((len(features), 3))
    p_test_x = np.zeros((len(test_features), 3))
    fold_aucs = []
    oof_x = []
    M_x = []
    P_x = []
    V_x = []
    j = 0
    target_id = target_mapping[target_name]
    drop_id = None
    if drop_name:
        drop_id = target_mapping[drop_name]
    i = target_id
    target_name = reverse_mapping[target_id]
    for fold, (tr_idx, val_idx) in tqdm(enumerate(skf.split(preds_tmp, target[target_name]))):
        X_train = preds_tmp.iloc[tr_idx].reset_index(drop=True)
        X_val = preds_tmp.iloc[val_idx].reset_index(drop=True)
        X_val_full = preds_tmp.iloc[val_idx].copy().reset_index(drop=True)
        train_target = target.iloc[tr_idx].reset_index(drop=True)
        val_target = target.iloc[val_idx].reset_index(drop=True)
        # filtering train and val!
        # masking third class
        if drop_id:
            drop_name = reverse_mapping[drop_id]
            train_mask_2 = (train_target[drop_name] == 0)
            val_mask_2 = (val_target[drop_name] == 0)
        else:
            train_mask_2 = (train_target[target_name] >= 0)
            val_mask_2 = (val_target[target_name] >= 0)
        X_train = X_train[train_mask_2]
        train_target = train_target[train_mask_2]
        X_val = X_val[val_mask_2]
        val_target = val_target[val_mask_2]
    
        val_ids = train_full[['id']].iloc[val_idx].reset_index(drop=True)
        y_train = train_target.iloc[:, i]
        y_val = val_target.iloc[:, i]
        X_test = test_tmp.copy()
    
        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_val_full.columns = X_val_full.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)
    
        print(val_target.columns[i])
        
        model = LogisticRegressionCV(max_iter=1000, random_state=seed, Cs=10, scoring='roc_auc', cv=5)
    
        model.fit(
            X_train, train_target.iloc[:, i]
        )
    
        p_val = model.predict_proba(X_val)[:, 1]
        p_val_full = model.predict_proba(X_val_full)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        p_val_x[val_idx, i] = p_val_full
        p_test_x[:, i] += p_test / N_FOLDS
    
        val_metric_x = roc_auc_score(val_target.iloc[:, i], p_val)
        print(val_metric_x)
        V_x.append(val_metric_x)
    
        p_val_x_2 = pd.DataFrame(p_val_x[val_idx, :], columns=target.columns)
        p_val_x_2['id'] = val_ids
        oof_x.append(p_val_x_2)
        M_x.append(val_metric_x)
    
        p_test_x_2 = pd.DataFrame(p_test_x)
        P_x.append(p_test_x_2)
        map_coef = pd.Series(model.coef_[0], index=model.feature_names_in_)
        print(map_coef.sort_values())
        if j < 4: del model
        del X_train, X_val, y_train, y_val
        gc.collect()
        j += 1
    
    oof_preds_1 = pd.concat(oof_x, ignore_index=True).sort_values(by='id').reset_index(drop=True)
    oof_preds_1.to_parquet(f'oof_preds_{target_id}_blend_{seed}.parquet')
    print(M_x)
    print(np.mean(M_x))
    p_test_x_final = np.mean(P_x, axis=0)
    test_preds_1 = pd.DataFrame(p_test_x_final, columns=target.columns)
    test_preds_1['id'] = test_ids
    test_preds_1.to_parquet(f'test_preds_{target_id}_blend_{seed}.parquet')
    h = pd.DataFrame(target.sum(), columns=['count']).astype(int)
    h['auc_val'] = np.round(np.mean(V_x, axis=0), 6)
    return oof_preds_1, test_preds_1

## 5. First model: fit vs at-risk 

In [ ]:
preds_tmp_1 = train_full[['id']]
preds_l_1 = []
test_l_1 = []
pred_col_1 = []
target_name = 'fit'
drop_name = 'unhealthy'
for model_type in MODEL_TYPES_1:
    for seed in SEED_LIST:
        run(model_type, target_name = target_name, drop_name = drop_name, seed = seed)
        preds_l_1.append(pd.read_parquet(f'oof_preds_1_{model_type}_{seed}.parquet')[[target_name]].rename(columns={target_name: f'my_base_{model_type}_seed_{seed}'}))
        test_l_1.append(pd.read_parquet(f'test_preds_1_{model_type}_{seed}.parquet')[[target_name]].rename(columns={target_name: f'my_base_{model_type}_seed_{seed}'}))
        pred_col_1.append(f'my_base_{model_type}_seed_{seed}')

preds_tmp_1 = pd.concat(preds_l_1, axis=1)
test_tmp_1 = pd.concat(test_l_1, axis=1)
preds_tmp_1 = to_logits(preds_tmp_1)
test_tmp_1 = to_logits(test_tmp_1)
preds_tmp_1.columns = pred_col_1
test_tmp_1.columns = pred_col_1

## 6. Average over seeds

In [ ]:
oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, preds_final_tmp = run_blend(preds_tmp_1, test_tmp_1, target_name = target_name, drop_name = drop_name, seed = seed)
    oof_x.append(oof_preds_tmp.set_index('id'))
    P_x.append(preds_final_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds_1 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_1.columns = target.columns
oof_preds_1['id'] = train_ids['id'].copy()
test_preds_1 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_1.columns = target.columns
test_preds_1['id'] = test_ids

## 7. Second model: unhealthy vs at-risk

In [ ]:
preds_tmp_2 = train_full[['id']]
preds_l_2 = []
test_l_2 = []
pred_col_2 = []
target_name = 'unhealthy'
drop_name = 'fit'
for model_type in MODEL_TYPES_2:
    for seed in SEED_LIST:
        run(model_type, target_name = target_name, drop_name = drop_name, seed = seed)
        preds_l_2.append(pd.read_parquet(f'oof_preds_2_{model_type}_{seed}.parquet')[[target_name]].rename(columns={target_name: f'my_base_{model_type}_seed_{seed}'}))
        test_l_2.append(pd.read_parquet(f'test_preds_2_{model_type}_{seed}.parquet')[[target_name]].rename(columns={target_name: f'my_base_{model_type}_seed_{seed}'}))
        pred_col_2.append(f'my_base_{model_type}_seed_{seed}')

preds_tmp_2 = pd.concat(preds_l_2, axis=1)
test_tmp_2 = pd.concat(test_l_2, axis=1)
preds_tmp_2 = to_logits(preds_tmp_2)
test_tmp_2 = to_logits(test_tmp_2)
preds_tmp_2.columns = pred_col_2
test_tmp_2.columns = pred_col_2

In [ ]:
oof_x = []
P_x = []
for seed in SEED_LIST:
    oof_preds_tmp, test_preds_tmp = run_blend(preds_tmp_2, test_tmp_2, target_name = target_name, drop_name = drop_name, seed = seed)
    oof_x.append(oof_preds_tmp.set_index('id'))
    P_x.append(test_preds_tmp.set_index('id'))
    train_ids = oof_preds_tmp[['id']]
oof_preds_2 = pd.DataFrame(np.mean(oof_x, axis=0))
oof_preds_2.columns = target.columns
oof_preds_2['id'] = train_ids['id'].copy()
test_preds_2 = pd.DataFrame(np.mean(P_x, axis=0))
test_preds_2.columns = target.columns
test_preds_2['id'] = test_ids

## 8. oof and test probabilities

In [ ]:
oof_preds = oof_preds_1[['id', 'fit']].join(oof_preds_2[['unhealthy']])
oof_preds = oof_preds.rename(columns={'fit': 'model_1', 'unhealthy': 'model_2'})
oof_preds['fit'] = oof_preds['model_1'] * (1 - oof_preds['model_2']) / (1 - oof_preds['model_1'] * oof_preds['model_2'])
oof_preds['unhealthy'] = oof_preds['model_2'] * (1 - oof_preds['model_1']) / (1 - oof_preds['model_1'] * oof_preds['model_2'])
oof_preds['at-risk'] = (1 - oof_preds['model_1']) * (1 - oof_preds['model_2']) / (1 - oof_preds['model_1'] * oof_preds['model_2'])

test_preds = test_preds_1[['id', 'fit']].join(test_preds_2[['unhealthy']])
test_preds = test_preds.rename(columns={'fit': 'model_1', 'unhealthy': 'model_2'})
test_preds['fit'] = test_preds['model_1'] * (1 - test_preds['model_2']) / (1 - test_preds['model_1'] * test_preds['model_2'])
test_preds['unhealthy'] = test_preds['model_2'] * (1 - test_preds['model_1']) / (1 - test_preds['model_1'] * test_preds['model_2'])
test_preds['at-risk'] = (1 - test_preds['model_1']) * (1 - test_preds['model_2']) / (1 - test_preds['model_1'] * test_preds['model_2'])

oof_preds = oof_preds[sorted(target_mapping.keys())].copy()
test_preds = test_preds[sorted(target_mapping.keys())].copy()

oof_preds.to_csv('oof_preds.csv', index=False)
test_preds.to_csv('test_preds.csv', index=False)

## 9. Probability calibration

In [ ]:
def public_preds(proba, bias):
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

def tune_bias(proba, y_true):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    history = [best_score]
    
    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_true, public_preds(proba, c))
                    if s > best_score + 1e-9:
                        best_bias, best_score, improved = c, s, True
                        history.append(best_score)
    return best_bias, best_score, history
    
raw_oof_ba = balanced_accuracy_score(y, np.argmax(oof_preds, axis=1))
print(f"📊 Raw OOF Balanced Accuracy: {raw_oof_ba:.6f}")

best_bias, tuned_ba, opt_history = tune_bias(oof_preds, y)
print(f"✨ Found Optimal Biases [0, 1, 2]: {best_bias}")
print(f"📈 Tuned OOF Balanced Accuracy: {tuned_ba:.6f} (+{tuned_ba - raw_oof_ba:.6f})")

In [ ]:
oof_final = public_preds(oof_preds, best_bias)
oof_labels = [reverse_mapping[p] for p in oof_final]

test_final = public_preds(test_preds, best_bias)
test_labels = [reverse_mapping[p] for p in test_final]

In [ ]:
recalls = recall_score(y, oof_final, average=None)

print(f"Recall {classes[0]}:    {recalls[0]:.4f}")
print(f"Recall {classes[1]}:    {recalls[1]:.4f}")
print(f"Recall {classes[2]}:    {recalls[2]:.4f}")

bal_acc = recalls.mean()
print(f"Balanced Accuracy: {bal_acc:.5f}")

In [ ]:
pd.Series(test_final).map(reverse_mapping).value_counts()

In [ ]:
cm = confusion_matrix(y, oof_final)
cm

In [ ]:
!rm *.parquet

In [ ]:
import datetime

ts = datetime.datetime.now().strftime("_%Y%m%d_%H-%M-%S")
ts = ''

sub = pd.DataFrame({ID: test_ids, TARGET: test_labels})
sub.to_csv(f'submission{ts}.csv', index=False)
sub.head()